# 02 · Inference with the packaged model
Loads `models/heart_model.joblib` (preprocessing + classifier in one sklearn Pipeline).

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path[:0] = [str(ROOT / 'src'), str(ROOT)]

In [2]:
import json
from heart.predict import load_model, load_metadata, predict_records
from heart import config
model = load_model(str(config.MODEL_FILE))
print(json.dumps(load_metadata(), indent=2)[:900])

{
  "model_name": "random_forest",
  "mlflow_run_id": "2055e2c2e9144566bf95a1728df42efb",
  "best_params": {
    "model__max_depth": null,
    "model__max_features": "sqrt",
    "model__min_samples_leaf": 5,
    "model__n_estimators": 200
  },
  "metrics": {
    "cv_accuracy_mean": 0.8423,
    "cv_precision_mean": 0.837,
    "cv_recall_mean": 0.8273,
    "cv_f1_mean": 0.8269,
    "cv_roc_auc_mean": 0.917,
    "cv_accuracy_std": 0.021,
    "cv_precision_std": 0.0681,
    "cv_recall_std": 0.0727,
    "cv_f1_std": 0.0228,
    "cv_roc_auc_std": 0.0286,
    "test_accuracy": 0.7869,
    "test_precision": 0.8,
    "test_recall": 0.7143,
    "test_f1": 0.7547,
    "test_roc_auc": 0.8885
  },
  "features": [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak",
    "ca",
    "sex",
    "fbs",
    "exang",
    "cp",
    "restecg",
    "slope",
    "thal"
  ],
  "trained_rows": 302,



In [3]:
patient = json.load(open(config.ROOT / 'sample' / 'patient.json'))
predict_records([patient], model)

[{'prediction': 1,
  'label': 'heart_disease',
  'probability_disease': 0.9948,
  'confidence': 0.9948,
  'risk_band': 'high'}]

Missing optional fields (`ca`, `thal`) are imputed by the pipeline:

In [4]:
p = dict(patient); p.pop('ca'); p['thal'] = None
predict_records([p], model)

[{'prediction': 1,
  'label': 'heart_disease',
  'probability_disease': 0.6682,
  'confidence': 0.6682,
  'risk_band': 'moderate'}]

In [5]:
import pandas as pd
pd.read_csv(config.ROOT / 'reports' / 'model_comparison.csv').set_index('name')[['cv_roc_auc_mean','cv_accuracy_mean','test_roc_auc','test_accuracy','test_recall']]

,cv_roc_auc_mean,cv_accuracy_mean,test_roc_auc,test_accuracy,test_recall
name,,,,,
logistic_regression,0.9088,0.8299,0.9210,0.8525,0.8214
random_forest,0.9170,0.8423,0.8885,0.7869,0.7143
hist_gradient_boosting,0.8945,0.8215,0.8961,0.8197,0.7143
